# Evaluación por ejecución
Se evalúan el baseline, ambos adapters y la reparación. El flujo selecciona el adapter por EX ponderada y guarda el análisis de errores y la predicción de generalización.

In [ ]:
from pathlib import Path
import os
raiz = Path.cwd() if (Path.cwd() / 'solucion.py').exists() else Path.cwd().parent
if not (raiz / 'solucion.py').exists():
    raise FileNotFoundError('No se encontró la raíz del repositorio')
os.chdir(raiz)
print(raiz)


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import os, signal, subprocess, sys
from pathlib import Path

def ejecutar_visible(comando, limite):
    proceso = subprocess.Popen(
        ['timeout', '--kill-after=10s', f'{limite}s', *comando],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True)
    try:
        for linea in proceso.stdout:
            print(linea, end='', flush=True)
        codigo = proceso.wait()
        if codigo:
            raise subprocess.CalledProcessError(codigo, comando)
    finally:
        if proceso.poll() is None:
            os.killpg(proceso.pid, signal.SIGTERM)

script = Path('scripts/colab_mysql.sh')
script.write_bytes(script.read_bytes().replace(b'\r\n', b'\n'))
ejecutar_visible(['bash', str(script)], 900)
os.environ['MYSQL_SOCKET'] = '/tmp/my8run/mysql.sock'


In [ ]:
!python ejecutar_proyecto.py --reanudar


In [ ]:
import pandas as pd
display(pd.read_csv("resultados/validacion_propia.csv"))
from IPython.display import Markdown, display
display(Markdown(Path("resultados/analisis.md").read_text(encoding="utf-8")))


## Resultados registrados

### Resultados medidos

| Configuración | Aciertos | EX ponderada | Tiempo medio (s) |
|---|---:|---:|---:|
| Base NF4 | 15/40 | 0.280 | 15.67 |
| QLoRA rango 8 | 21/40 | 0.439 | 11.49 |
| QLoRA rango 16 | 20/40 | 0.402 | 11.83 |
| Adapter con reparación | 21/40 | 0.427 | 13.20 |
| Entrega seleccionada | 21/40 | 0.439 | 11.49 |

Se seleccionó lora_r8, con reparación desactivada. La EX oculta estimada es 0.35; corresponde a una predicción, no a una medición oficial.

La carga del modelo tardó 6.37 segundos. Los tres casos repetidos cumplieron determinismo, consulta de solo lectura, límite de llamadas y tiempo. Las 23 pruebas automatizadas se aprobaron.

[Análisis completo](../resultados/analisis.md)